# 🏗️ SOLVENCY II CAPITAL MODELLING — Complete Hands-On Project

**Dataset:** freMTPL2 (French Motor Third Party Liability — 678K policies, 26K claims)  
**Goal:** Build a full pricing → capital pipeline that you can explain in a Solvency II interview  
**Approach:** Every step has instructions. YOU write the code. Each step < 1 minute.  

---

## What This Project Covers (End-to-End)

| Module | What You Build | FRM Connection | Solvency II Connection |
|--------|---------------|----------------|------------------------|
| 1 | Data loading & EDA | — | Understanding the risk exposure |
| 2 | Frequency GLM (Poisson) | Poisson distribution | Premium Risk — expected claim count |
| 3 | Severity GLM (Gamma) | Lognormal/Gamma tails | Premium Risk — expected claim size |
| 4 | Pure Premium calculation | Expected value | Technical pricing = E[Loss] |
| 5 | Aggregate Loss Distribution | Compound distributions | The distribution that generates SCR |
| 6 | Monte Carlo Simulation + VaR/ES | VaR, CVaR, Monte Carlo | Core of internal capital model |
| 7 | Copulas & Dependency | Gaussian copula, t-copula | Diversification benefit in SCR |
| 8 | Solvency II Standard Formula | Regulatory capital | SCR calculation |
| 9 | Internal Model vs Standard Formula | Model risk | Why internal models exist |
| 10 | Capital Allocation & RORAC | Risk-adjusted performance | Capital cost → pricing feedback |

---

# MODULE 1: DATA LOADING & EXPLORATION

**Why this matters for Solvency II:** Before modelling capital, you must understand your EXPOSURE. The SCR is driven by the VOLUME of business (earned premium, number of policies) and the RISK PROFILE (what mix of vehicles, drivers, regions). This module teaches you to quantify both.

---

## Step 1.1: Import Libraries

**What to do:** Import `pandas`, `numpy`, `matplotlib.pyplot`, and `seaborn`.  
**Why:** These are your workhorses. pandas for data, numpy for math, matplotlib/seaborn for visuals.  
**Also:** Set `%matplotlib inline` and `sns.set_style('whitegrid')` for clean plots.  
**Also:** `import warnings; warnings.filterwarnings('ignore')` to keep output clean.

In [ ]:
# YOUR CODE: Import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
# Set inline plotting, whitegrid style, suppress warnings

## Step 1.2: Load the Frequency Dataset

**What to do:** Read `freMTPL2freq.csv` from `C:\Users\226862\Documents\InsPricing\freMTPL2_data\`  
**How:** `pd.read_csv(path)` — store it in a variable called `freq_df`  
**Then:** Print `.shape` and `.head(3)` to see what you've loaded.  

**What the columns mean:**
- `IDpol`: Policy ID (unique identifier)
- `ClaimNb`: Number of claims this policy had (0, 1, 2, ...)
- `Exposure`: Fraction of year the policy was active (0.5 = 6 months)
- `VehPower`: Vehicle power category (4-15)
- `VehAge`: Age of vehicle in years
- `DrivAge`: Age of driver in years
- `BonusMalus`: No-claim bonus/malus coefficient (50=best, >100=bad history)
- `VehBrand`: Vehicle brand category
- `VehGas`: Fuel type (Regular/Diesel)
- `Area`: Population density category (A-F, A=rural, F=urban)
- `Density`: Population density (inhabitants/km²)
- `Region`: Geographic region of France

In [ ]:
# YOUR CODE: Load freq data, print shape and head

## Step 1.3: Load the Severity Dataset

**What to do:** Read `freMTPL2sev.csv` — store as `sev_df`  
**Then:** Print shape and head.  

**What it contains:** Each row = one claim payment. `IDpol` links back to the policy. `ClaimAmount` = how much was paid for that claim.  

**Important:** One policy can have multiple claim rows (if they had >1 claim).

In [ ]:
# YOUR CODE: Load severity data, print shape and head

## Step 1.4: Merge the Datasets

**What to do:** Merge `freq_df` and `sev_df` on `IDpol` using a LEFT join.  
**Why LEFT join:** We want ALL policies (even those with 0 claims). Severity only has rows for policies WITH claims.  
**How:** `df = freq_df.merge(sev_df, on='IDpol', how='left')`  
**Then:** For policies with no claims, `ClaimAmount` will be NaN. Fill it with 0: `df['ClaimAmount'].fillna(0, inplace=True)`  
**Check:** Print `df.shape` — should have more rows than freq_df (because some policies have multiple claims).

**⚠️ IMPORTANT:** After merge, some policies with 2+ claims will appear multiple times. For FREQUENCY modelling later, we'll use `freq_df` directly. For SEVERITY modelling, we'll use `sev_df`. The merge is for understanding the full picture.

In [ ]:
# YOUR CODE: Merge datasets, fill NaN claim amounts with 0, check shape

## Step 1.5: Basic Summary Statistics

**What to do:** On `freq_df`, compute and print:
1. Total number of policies: `len(freq_df)`
2. Total exposure (policy-years): `freq_df['Exposure'].sum()`
3. Total number of claims: `freq_df['ClaimNb'].sum()`
4. Overall claim frequency: total claims / total exposure
5. % of policies with 0 claims: `(freq_df['ClaimNb'] == 0).mean()`

**Why this matters for Solvency II:** These are your VOLUME measures. The Standard Formula SCR = f(Volume, σ). Volume here = exposure × premium rate. The claim frequency tells you if this is a high-frequency/low-severity book (motor usually is).

**Expected results:** Frequency should be around 6-8% (typical for French motor MTPL).

In [ ]:
# YOUR CODE: Calculate and print 5 summary statistics

## Step 1.6: Claim Count Distribution

**What to do:** Create a frequency table of `ClaimNb`: `freq_df['ClaimNb'].value_counts().sort_index()`  
**Then:** Make a bar chart of the claim count distribution.  

**What to notice:**
- Most policies have 0 claims (~93%)
- Very few have 2+ claims
- This is classic **Poisson-like** shape (spike at 0, rapid decay)

**FRM Connection:** You studied the Poisson distribution. If claims are independent events occurring at a constant rate, the count follows Poisson. The high proportion of zeros is natural when λ (expected frequency) is small.

**Interview line:** "The claim count distribution is consistent with a Poisson process — approximately 93% zeros, decay consistent with λ ≈ 0.07."

In [ ]:
# YOUR CODE: Value counts of ClaimNb, then bar chart

## Step 1.7: Severity Distribution — Histogram + Stats

**What to do:** 
1. Print `sev_df['ClaimAmount'].describe()` — look at mean, median, max
2. Plot a histogram of `ClaimAmount` — use `bins=100` and set `xlim(0, 20000)` to see the bulk
3. Plot a LOG histogram: `np.log(sev_df['ClaimAmount']).hist(bins=50)`

**What to notice:**
- Mean >> Median → RIGHT SKEWED (heavy tail)
- On log-scale, it looks roughly Normal → the raw data is LOGNORMAL
- The max claim could be enormous (100K+)

**FRM Connection:** You studied lognormal distributions. If ln(X) ~ Normal, then X ~ Lognormal. This is exactly what severity looks like. The RIGHT TAIL is what drives capital — a few huge claims dominate the 99.5th percentile.

**Interview line:** "Claim severity exhibits lognormal characteristics — log-transformed amounts are approximately normal. The heavy right tail means a small number of large claims disproportionately drives the capital requirement."

In [ ]:
# YOUR CODE: Describe severity, histogram on raw scale, histogram on log scale

## Step 1.8: Exposure Distribution Check

**What to do:** Plot histogram of `Exposure`. Print mean and median.  

**Why:** Exposure is the OFFSET in your GLM. If a policy was only active for 6 months, it had half the chance to have a claim. We need to understand if most policies are full-year (exposure ≈ 1) or partial.

**Solvency II connection:** Premium volume = sum of earned premium = approximately sum(Exposure × rate). If many policies are partial-year, your earned premium is less than written premium.

In [ ]:
# YOUR CODE: Histogram of exposure, print mean and median

## Step 1.9: Key Risk Factor — BonusMalus

**What to do:**
1. Print `freq_df['BonusMalus'].describe()`
2. Plot histogram of BonusMalus
3. Compute claim frequency BY BonusMalus bucket: group into (50-60, 60-80, 80-100, 100-150, 150+) and compute claims/exposure for each

**What to notice:** Higher BonusMalus = worse driving history = MUCH higher frequency. This is the single most predictive variable in motor insurance.

**Hint for grouping:** `pd.cut(freq_df['BonusMalus'], bins=[49, 60, 80, 100, 150, 250])`

**Interview line:** "BonusMalus is the strongest predictor — a coefficient of 150 implies roughly 3-4x the frequency of a coefficient of 50. It captures moral hazard and adverse selection directly."

In [ ]:
# YOUR CODE: BonusMalus stats, histogram, frequency by BM bucket

## Step 1.10: Key Risk Factor — Driver Age

**What to do:**
1. Create age bands: `pd.cut(freq_df['DrivAge'], bins=[17, 22, 30, 45, 65, 100])`
2. For each age band, compute: total claims / total exposure = frequency
3. Plot as a bar chart

**What to notice:** Young drivers (18-22) have MUCH higher frequency. Classic U-shape or decreasing pattern.

**Why for Capital:** Young drivers contribute disproportionately to risk. If your portfolio has more young drivers, you need more capital. This is the "risk profile" that differentiates Standard Formula from Internal Model.

In [ ]:
# YOUR CODE: Age bands, frequency by band, bar chart

## Step 1.11: Key Risk Factor — Area (Urban vs Rural)

**What to do:**
1. Group by `Area` (A through F)
2. Compute frequency (claims/exposure) for each area
3. Also compute average claim severity by area (merge sev_df, group by Area)
4. Plot both as bar charts (side by side: `fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))`)

**What to notice:** Urban areas (E, F) have higher frequency AND higher severity (more traffic, higher repair costs).

**Capital insight:** Geographic concentration matters. If 80% of your book is in Area F (Paris), you're NOT diversified — a local event (flood, protest, new speed limits) affects your entire portfolio.

In [ ]:
# YOUR CODE: Frequency and severity by Area, side-by-side plots

## Step 1.12: Summary — Portfolio Profile Table

**What to do:** Create a summary DataFrame with one row per key variable showing:
- Variable name
- Number of unique values / range
- Frequency range (min to max across categories)
- "Risk leverage" (max frequency / min frequency)

**Example output format:**
```
Variable     | Levels | Freq Range      | Risk Leverage
BonusMalus   | ~100   | 2% - 25%        | 12.5x
DrivAge      | 5 bands| 5% - 15%        | 3.0x
Area         | 6      | 4% - 10%        | 2.5x
VehGas       | 2      | 6% - 8%         | 1.3x
```

**Why:** This tells you which variables MATTER most for capital. High risk leverage = high parameter sensitivity = high contribution to capital uncertainty.

**Interview line:** "I profiled the portfolio across all risk factors. BonusMalus has the highest risk leverage at ~12x, meaning the worst segment has 12 times the frequency of the best. This variable alone drives significant capital differentiation between segments."

In [ ]:
# YOUR CODE: Create portfolio profile summary table

---
# MODULE 2: FREQUENCY MODELLING (Poisson GLM)

**Why this matters for Solvency II:** The frequency model gives you λᵢ (expected claim rate for each policy). This feeds DIRECTLY into the Monte Carlo capital model — you simulate N ~ Poisson(Σλᵢ) for the portfolio aggregate. A wrong frequency model → wrong mean → wrong VaR → wrong SCR.

**FRM Connection:** You studied Poisson distribution properties: E[N] = Var[N] = λ. The GLM lets you make λ depend on risk factors: λᵢ = Exposureᵢ × exp(β₀ + β₁x₁ + ...)

---

## Step 2.1: Prepare Data for Modelling

**What to do:**
1. Cap `ClaimNb` at 4 (truncate extreme values): `freq_df['ClaimNb'] = freq_df['ClaimNb'].clip(upper=4)`
2. Cap `VehAge` at 20: `freq_df['VehAge'] = freq_df['VehAge'].clip(upper=20)`  
3. Cap `DrivAge` at 90: `freq_df['DrivAge'] = freq_df['DrivAge'].clip(upper=90)`
4. Cap `BonusMalus` at 150: `freq_df['BonusMalus'] = freq_df['BonusMalus'].clip(upper=150)`
5. Cap `Density` at 25000: `freq_df['Density'] = freq_df['Density'].clip(upper=25000)`
6. Take log of Density: `freq_df['LogDensity'] = np.log(freq_df['Density'])`

**Why:** Outliers destabilize GLMs. Capping at reasonable values prevents one extreme observation from dominating. Log of density because the relationship with frequency is likely multiplicative, not additive.

**Why for Capital:** Unstable parameters → higher parameter uncertainty → higher capital. Data cleaning reduces artificial uncertainty.

In [ ]:
# YOUR CODE: Cap variables and create LogDensity

## Step 2.2: Create Train/Test Split

**What to do:**
1. Import `from sklearn.model_selection import train_test_split`
2. Split `freq_df` into 80% train, 20% test: `train_freq, test_freq = train_test_split(freq_df, test_size=0.2, random_state=42)`
3. Print sizes of both sets

**Why:** We'll fit the model on train and VALIDATE on test. This proves the model generalizes — crucial for capital models which must be validated per Solvency II Pillar 2 requirements.

**Interview line:** "The internal model must be validated — I used an 80/20 holdout to demonstrate out-of-sample predictive power, as required under Solvency II's Use Test."

In [ ]:
# YOUR CODE: Train/test split, print sizes

## Step 2.3: Define Features (X) and Target (y)

**What to do:**
1. Define feature columns: `['VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'LogDensity', 'VehGas', 'Area', 'Region']`
2. Separate numeric features: `['VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'LogDensity']`
3. Separate categorical features: `['VehGas', 'Area', 'Region']`
4. Target: `y_train = train_freq['ClaimNb']`
5. Exposure (used as offset): `exposure_train = train_freq['Exposure']`

**Why Exposure is special:** In Poisson regression, we model: E[Claims] = Exposure × exp(Xβ). The `log(Exposure)` enters as an OFFSET (fixed coefficient = 1). This means: a policy active for 6 months has half the expected claims of a full-year policy, all else equal.

**FRM connection:** This is rate modelling. We're estimating the INTENSITY of the Poisson process, not the raw count.

In [ ]:
# YOUR CODE: Define feature lists, extract y and exposure from train set

## Step 2.4: One-Hot Encode Categoricals

**What to do:**
1. Use `pd.get_dummies()` on the categorical columns, with `drop_first=True` (to avoid multicollinearity)
2. Combine with numeric features into one DataFrame called `X_train`
3. Do the same for test set → `X_test`

**How:**
```python
X_train = pd.get_dummies(train_freq[feature_cols], columns=cat_features, drop_first=True)
X_test = pd.get_dummies(test_freq[feature_cols], columns=cat_features, drop_first=True)
# Align columns (test might be missing some dummies):
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
```

**Why drop_first:** One category becomes the "reference" level. All coefficients are relative to that baseline. Without dropping, the design matrix is rank-deficient.

Print `X_train.shape` to see how many features you have.

In [ ]:
# YOUR CODE: One-hot encode, align columns, print shape

## Step 2.5: Fit Poisson GLM using statsmodels

**What to do:**
1. `import statsmodels.api as sm`
2. Add constant: `X_train_sm = sm.add_constant(X_train)`
3. Fit Poisson GLM with log-link and exposure offset:
```python
freq_model = sm.GLM(
    y_train,
    X_train_sm,
    family=sm.families.Poisson(link=sm.families.links.Log()),
    offset=np.log(exposure_train)
).fit()
```
4. Print `freq_model.summary()` — look at the coefficients

**What the output tells you:**
- Positive coefficient → increases claim frequency
- BonusMalus should be strongly positive (higher BM → more claims)
- Young driver ages should have higher frequency (captured by DrivAge coefficient)
- p-values tell you which variables are statistically significant

**FRM connection:** This is maximum likelihood estimation (MLE) of the Poisson parameter. The log-link ensures λ > 0 always.

In [ ]:
# YOUR CODE: Fit Poisson GLM with statsmodels, print summary

## Step 2.6: Interpret Key Coefficients

**What to do:**
1. Extract the BonusMalus coefficient: `bm_coef = freq_model.params['BonusMalus']`
2. Compute the MULTIPLICATIVE effect: `np.exp(bm_coef)` — this is the factor by which frequency changes per unit increase in BonusMalus
3. Example: if exp(coef) = 1.02, then each 1-unit increase in BM multiplies frequency by 1.02. Going from BM=50 to BM=100 = (1.02)^50 = 2.7x the frequency!
4. Do the same for DrivAge and LogDensity

**Print a nice interpretation:**
```
BonusMalus: +1 unit → frequency multiplied by {exp(coef):.4f}
  Going from BM=50 to BM=100 → frequency multiplied by {exp(coef*50):.2f}
```

**Interview line:** "The Poisson GLM shows BonusMalus has an exponential effect on frequency — each unit increase multiplies the hazard rate by approximately 2%. A policy with BM=100 has roughly 2.7x the expected frequency of a BM=50 policy, all else equal."

In [ ]:
# YOUR CODE: Extract and interpret key coefficients

## Step 2.7: Predict on Test Set & Validate

**What to do:**
1. `X_test_sm = sm.add_constant(X_test)`
2. Predict: `pred_freq = freq_model.predict(X_test_sm, offset=np.log(test_freq['Exposure']))`
3. Compare TOTAL: `predicted total = pred_freq.sum()` vs `actual total = test_freq['ClaimNb'].sum()`
4. Compute prediction ratio: predicted/actual — should be close to 1.0

**Why this matters for Solvency II:** Regulatory validation requires demonstrating the model predicts well out-of-sample. A prediction ratio between 0.98 and 1.02 is excellent.

**Also:** Group by decile of predicted frequency and check if actual frequency increases monotonically ("lift chart" validation). High-predicted-risk policies should have high actual claims.

In [ ]:
# YOUR CODE: Predict on test set, compare total predicted vs actual

## Step 2.8: Portfolio-Level Predicted Frequency

**What to do:**
1. Predict on the FULL dataset: get predicted frequency for every policy
2. Store it: `freq_df['pred_freq'] = freq_model.predict(X_full_sm, offset=np.log(freq_df['Exposure']))`
3. Compute PORTFOLIO expected claims: `total_expected_claims = freq_df['pred_freq'].sum()`
4. Print it: this is your λ_portfolio for the Monte Carlo simulation later!

**Bridge to Capital:** This `total_expected_claims` becomes the Poisson parameter in your aggregate loss simulation. You'll simulate N ~ Poisson(λ_portfolio) later.

In [ ]:
# YOUR CODE: Predict on full dataset, compute portfolio total expected claims

---
# MODULE 3: SEVERITY MODELLING (Gamma GLM)

**Why this matters for Solvency II:** The severity distribution's TAIL drives capital. A thin-tailed severity (Exponential) → low capital. A fat-tailed severity (Pareto) → high capital. Getting the tail right is the difference between adequate and inadequate reserves.

**FRM Connection:** You studied Gamma, Lognormal, and Pareto distributions. Gamma GLM is standard for severity (positive, right-skewed, variance proportional to mean²).

---

## Step 3.1: Prepare Severity Data

**What to do:**
1. Start from `sev_df` merged with policy features: `sev_data = sev_df.merge(freq_df[['IDpol'] + feature_cols_list], on='IDpol', how='left')`
2. Remove any rows where ClaimAmount ≤ 0 (Gamma needs strictly positive values)
3. Cap extreme claims at the 99.5th percentile: `cap = sev_data['ClaimAmount'].quantile(0.995)` then clip
4. Print: number of claims, mean severity, median severity, 99.5th percentile

**Why cap at 99.5%:** Extreme claims (>$100K) often get special treatment in capital models (modelled separately with EVT/Pareto). For the "attritional" severity model, we cap to get stable Gamma estimates.

**Interview line:** "I split the severity into attritional (below the 99.5th percentile, modelled with Gamma GLM) and large claims (above threshold, modelled with GPD). This ensures the body and tail of the distribution are each captured by appropriate models."

In [ ]:
# YOUR CODE: Prepare severity data — merge, remove zeros, cap, summarize

## Step 3.2: Fit Gamma GLM for Severity

**What to do:**
1. Prepare X matrix from severity data (same features as frequency, one-hot encoded)
2. Target: `y_sev = sev_data['ClaimAmount']`
3. Fit Gamma GLM with LOG link:
```python
sev_model = sm.GLM(
    y_sev,
    X_sev_sm,
    family=sm.families.Gamma(link=sm.families.links.Log())
).fit()
```
4. Print summary

**Why Gamma with log link:** Gamma assumes Var[Y] ∝ E[Y]² — variance increases with the mean. This is true for insurance claims: expensive claims are also more variable. Log link ensures predicted severity > 0.

**What to look for:** Fewer variables may be significant for severity (severity is harder to predict than frequency). That's OK — it still gives us E[X|covariates].

In [ ]:
# YOUR CODE: Fit Gamma GLM for severity, print summary

## Step 3.3: Extract Key Severity Parameters for Capital Model

**What to do:**
1. Get the Gamma shape parameter (also called alpha): `alpha = 1 / sev_model.scale`
2. Get the predicted mean severity across the portfolio: `mean_sev_portfolio = sev_model.predict(X_sev_sm).mean()`
3. Compute the CV (coefficient of variation) of severity: `cv = 1 / np.sqrt(alpha)`
4. Print all three

**Why these matter for capital:**
- `mean_sev_portfolio` → E[X] in your aggregate loss model
- `cv` → drives the width of the aggregate distribution. Higher CV → heavier tail → more capital
- For Gamma: CV = 1/√α. If α = 1, CV = 1 (very dispersed). If α = 4, CV = 0.5 (tighter).

**FRM connection:** The Gamma distribution's shape parameter controls tail heaviness. Low α = heavy tail (close to Exponential). High α = light tail (approaches Normal).

In [ ]:
# YOUR CODE: Extract alpha, mean severity, CV

## Step 3.4: Validate Severity Model — QQ Plot

**What to do:**
1. Compute residuals: `residuals = y_sev / sev_model.predict(X_sev_sm)` (these are "response residuals" for Gamma)
2. Plot a QQ plot against Gamma distribution: `from scipy import stats; stats.probplot(residuals, dist='gamma', sparams=(alpha,), plot=plt)`
3. Also plot histogram of residuals — should be roughly Gamma-shaped (right-skewed, mode < mean)

**What to look for:** If the QQ plot deviates at the upper end (observed > theoretical), the Gamma tail is TOO THIN — you need a heavier-tailed model (Pareto/GPD) for the extreme claims.

**Interview line:** "I validated the Gamma severity model with QQ plots. The body fits well, but the upper tail deviates — indicating that for capital modelling purposes, I need to supplement with a GPD tail model for claims above the threshold."

In [ ]:
# YOUR CODE: QQ plot of severity residuals

## Step 3.5: Fit GPD for the Tail (Extreme Value Theory)

**What to do:**
1. Choose a threshold: use the 95th percentile of ClaimAmount: `threshold = sev_df['ClaimAmount'].quantile(0.95)`
2. Get excesses: `excesses = sev_df.loc[sev_df['ClaimAmount'] > threshold, 'ClaimAmount'] - threshold`
3. Fit GPD: `from scipy.stats import genpareto; shape, loc, scale = genpareto.fit(excesses, floc=0)`
4. Print: threshold, number of exceedances, shape (ξ), scale (σ)
5. The shape parameter ξ tells you tail heaviness: ξ > 0 = heavy tail (Pareto-like)

**FRM Connection:** This is EXACTLY the EVT/POT (Peaks Over Threshold) method from FRM Part 1! You select a threshold u, fit GPD to excesses Y = X - u, and the shape parameter ξ determines tail behavior.

**Interview line:** "For the extreme tail of severity, I applied the Peaks Over Threshold approach from EVT. The fitted GPD shape parameter ξ = [your value] indicates a [heavy/moderate] tail. This is critical for the 99.5th percentile — the Gamma body model alone would underestimate capital by approximately [X]%."

In [ ]:
# YOUR CODE: Fit GPD to severity tail

---
# MODULE 4: PURE PREMIUM & TECHNICAL PRICING

**Why this matters for Solvency II:** The pure premium = E[Loss] = the MEAN of your aggregate distribution. The SCR is about how far the 99.5th percentile is from this mean. Without the correct mean, your capital model is anchored to the wrong point.

**Connection:** Pure Premium = Frequency × Severity = E[N] × E[X] per unit of exposure.

---

## Step 4.1: Compute Pure Premium per Policy

**What to do:**
1. For each policy in `freq_df`, you already have `pred_freq` (from Module 2)
2. Get predicted severity for each policy (use the Gamma model — predict on freq_df features)
3. Pure premium = pred_freq × pred_severity (for each policy)
4. Store as `freq_df['pure_premium']`
5. Print: total portfolio pure premium = `freq_df['pure_premium'].sum()`

**Note:** For the severity prediction, you need to apply the Gamma model to the freq_df features. Use the mean prediction from the severity model as a portfolio-level average if alignment is tricky.

**Simple approach:** `freq_df['pure_premium'] = freq_df['pred_freq'] * mean_sev_portfolio`

**Why:** The total pure premium = expected total loss for the portfolio = the MEAN of your aggregate loss distribution.

In [ ]:
# YOUR CODE: Compute pure premium per policy and total

## Step 4.2: Compute Expected Loss Ratio

**What to do:**
1. Assume an average premium rate. For this exercise, let's say the company charges a 30% loading:
   `gross_premium_total = freq_df['pure_premium'].sum() * 1.30`
2. Expected loss ratio = pure premium / gross premium = 1/1.30 = 76.9%
3. Print: total expected claims, total gross premium, expected loss ratio

**Why for Solvency II Standard Formula:**
- Premium Volume (V_prem) = gross premium earned
- The expected loss ratio is used in BF-type methods and as a validation
- If actual LR >> expected LR → premium risk is materializing → SCR breach possible

**Interview line:** "The technical loss ratio for this portfolio is approximately 77%, implying a 30% margin for expenses and capital costs. The Solvency II premium risk SCR essentially asks: how far could the actual loss ratio deviate from this 77% in a 1-in-200 year?"

In [ ]:
# YOUR CODE: Compute gross premium and expected loss ratio

## Step 4.3: Loss Ratio Volatility — The Key Capital Input

**What to do:**
1. Simulate multiple "years" of claims using your frequency model:
   - For 1000 iterations: draw N ~ Poisson(total_expected_claims), draw N severities, compute total loss
2. Compute the loss ratio for each simulated year: `sim_LR = sim_total_loss / gross_premium`
3. Compute the standard deviation of simulated loss ratios: `sigma_LR = sim_LRs.std()`
4. Print: mean LR, std LR, CoV of LR

**Why:** This σ_LR IS the premium risk volatility parameter. In the Standard Formula: SCR_premium ≈ 3 × σ × Volume.

**Quick simulation (you'll do full Monte Carlo later):**
```python
sim_losses = []
for _ in range(1000):
    n = np.random.poisson(total_expected_claims)
    claims = np.random.gamma(alpha, mean_sev_portfolio/alpha, size=n)
    sim_losses.append(claims.sum())
```

In [ ]:
# YOUR CODE: Simulate 1000 years, compute loss ratio volatility

---
# MODULE 5: AGGREGATE LOSS DISTRIBUTION

**Why this matters for Solvency II:** The aggregate loss distribution IS what capital modelling is about. You need its 99.5th percentile. This module builds the compound Poisson-Gamma distribution properly.

**FRM Connection:** S = X₁ + X₂ + ... + X_N where N ~ Poisson(λ) and Xᵢ ~ Gamma(α, β). You studied this as the "compound distribution." E[S] = E[N]×E[X], Var[S] = E[N]×Var[X] + Var[N]×E[X]².

---

## Step 5.1: Analytical Moments of Aggregate Loss

**What to do:** Compute the theoretical moments WITHOUT simulation first:
1. E[S] = E[N] × E[X] = λ_portfolio × mean_severity
2. Var[S] = E[N] × Var[X] + Var[N] × E[X]²
   - For Poisson: Var[N] = E[N] = λ
   - For Gamma: Var[X] = E[X]² / α = mean_sev² / α
   - So: Var[S] = λ × (mean_sev² / α) + λ × mean_sev² = λ × mean_sev² × (1/α + 1)
3. SD[S] = sqrt(Var[S])
4. CV[S] = SD[S] / E[S]
5. Print all: E[S], SD[S], CV[S]

**Why compute analytically first:** You'll validate your Monte Carlo against these. If your simulation mean doesn't match E[S], something is wrong with the code.

**Interview line:** "I first computed analytical moments of the compound Poisson-Gamma distribution to validate my simulation. The coefficient of variation of the aggregate loss is [X]%, which is consistent with a motor portfolio of this size."

In [ ]:
# YOUR CODE: Compute analytical E[S], Var[S], SD[S], CV[S]

## Step 5.2: Full Monte Carlo — 100,000 Simulations

**What to do:**
1. Set `n_sim = 100_000` and `np.random.seed(42)`
2. Create empty array: `aggregate_losses = np.zeros(n_sim)`
3. Loop:
   ```python
   for i in range(n_sim):
       n_claims = np.random.poisson(lambda_portfolio)
       if n_claims > 0:
           severities = np.random.gamma(alpha, mean_sev / alpha, size=n_claims)
           aggregate_losses[i] = severities.sum()
   ```
4. Print: mean, std, compare to analytical values from Step 5.1

**NOTE:** `np.random.gamma(shape, scale, size)` — where shape=α, scale=mean/α=β.

**Validation:** Your simulated mean should be within 1% of the analytical E[S]. If not, check your parameters.

**This takes ~30 seconds to run. That's fine.**

In [ ]:
# YOUR CODE: Monte Carlo simulation of aggregate losses

## Step 5.3: Visualize the Aggregate Loss Distribution

**What to do:**
1. Plot histogram of `aggregate_losses` with `bins=200`
2. Add vertical lines at:
   - Mean (green, dashed): `plt.axvline(mean, color='green', linestyle='--', label='Mean (E[S])')`
   - VaR 99.5% (red, solid): `plt.axvline(np.percentile(aggregate_losses, 99.5), color='red', label='VaR 99.5%')`
   - VaR 99.0% (orange, dashed)
3. Add legend, title: "Aggregate Loss Distribution — Motor MTPL Portfolio"

**What to observe:** The gap between the green line (mean) and red line (VaR) IS your SCR. Visually, you can see that the distribution is right-skewed — the 99.5th percentile is far from the mean.

**Interview line:** "The aggregate loss distribution is right-skewed with a coefficient of variation of [X]%. The 99.5th percentile is approximately [Y]x the mean, reflecting the compound Poisson-Gamma tail behavior."

In [ ]:
# YOUR CODE: Histogram with VaR and mean lines

## Step 5.4: Compare Gamma Severity vs Lognormal Severity

**What to do:**
1. Re-run the simulation but with LOGNORMAL severity instead of Gamma:
   - Match the same mean and CV as the Gamma
   - Lognormal params: `sigma_ln = np.sqrt(np.log(1 + cv**2))`, `mu_ln = np.log(mean_sev) - 0.5*sigma_ln**2`
   - Draw: `np.random.lognormal(mu_ln, sigma_ln, size=n_claims)`
2. Compare the 99.5th percentile: Gamma vs Lognormal
3. Print both VaR(99.5%) values

**What to observe:** Lognormal has a HEAVIER TAIL than Gamma (for same mean and CV). So VaR(99.5%) will be higher for Lognormal. This demonstrates why severity distribution choice directly impacts capital.

**FRM connection:** You studied that different distributions with the same first two moments can have VERY different tails. This is exactly that concept applied to capital.

**Interview line:** "The choice of severity distribution materially impacts the SCR. With identical mean and variance, the Lognormal assumption gives approximately [X]% higher capital than the Gamma assumption, due to its heavier tail. This is why model validation — including QQ plots and tail goodness-of-fit — is essential."

In [ ]:
# YOUR CODE: Simulate with lognormal severity, compare VaR to Gamma version

---
# MODULE 6: VALUE AT RISK & EXPECTED SHORTFALL

**Why this matters for Solvency II:** SCR = VaR(99.5%) − E[Loss]. This module computes your VaR, ES, and explores how confidence level choice affects capital.

**FRM Connection:** You studied VaR properties (not sub-additive), ES properties (sub-additive, coherent), confidence intervals for VaR estimates, backtesting. ALL of this applies directly.

---

## Step 6.1: Compute VaR at Multiple Confidence Levels

**What to do:**
1. From your `aggregate_losses` simulation, compute:
   - VaR(90%) = `np.percentile(aggregate_losses, 90)`
   - VaR(95%)
   - VaR(99%)
   - VaR(99.5%) ← This is the Solvency II level
   - VaR(99.9%)
2. Print a table showing: confidence level, VaR value, VaR/Mean ratio, implied "return period" (1/(1-α))

**Example output:**
```
Confidence | VaR ($M)  | VaR/Mean | Return Period
90.0%      | 42.1      | 1.15x    | 1-in-10 year
99.5%      | 55.3      | 1.51x    | 1-in-200 year  ← SCR
```

**Interview line:** "At the 99.5% confidence level (1-in-200 year), the aggregate loss is approximately [X] times the expected loss. This means our capital buffer must be [X-1] times the expected loss to survive the worst-case scenario."

In [ ]:
# YOUR CODE: Compute VaR at multiple levels, create table

## Step 6.2: Compute Expected Shortfall (CVaR)

**What to do:**
1. ES(99.5%) = average of all simulated losses ABOVE VaR(99.5%)
   ```python
   var_995 = np.percentile(aggregate_losses, 99.5)
   es_995 = aggregate_losses[aggregate_losses >= var_995].mean()
   ```
2. Also compute ES at 99% and 99.9%
3. Print: VaR(99.5%), ES(99.5%), ratio ES/VaR

**Why ES matters:**
- VaR tells you: "we won't lose more than $X with 99.5% probability"
- ES tells you: "IF we DO lose more than VaR, the AVERAGE loss is $Y"
- ES captures HOW BAD the tail is, not just where it starts
- ES is SUB-ADDITIVE (diversification always helps) unlike VaR

**FRM connection:** You studied that ES is a "coherent" risk measure (satisfies sub-additivity, monotonicity, positive homogeneity, translation invariance). VaR fails sub-additivity. Swiss Solvency Test uses ES instead of VaR.

**Interview line:** "I compute both VaR and ES. For this portfolio, ES(99.5%) is approximately [X]% higher than VaR(99.5%), indicating meaningful tail risk beyond the regulatory threshold. I'd use ES for internal risk management and reinsurance purchasing decisions, even though Solvency II prescribes VaR."

In [ ]:
# YOUR CODE: Compute ES at multiple levels, compare to VaR

## Step 6.3: SCR Calculation

**What to do:**
1. SCR = VaR(99.5%) − E[S] (capital above expected loss)
2. Also compute: SCR as % of expected loss
3. Also compute: SCR as % of gross premium (the "capital intensity")
4. Print all three

**Why subtract the mean:** The expected loss is already priced into the premium. Capital is for the UNEXPECTED excess. You only need capital for the gap between what you expect and what could happen.

**Typical ranges for motor:** SCR/Premium = 10-20% for diversified motor portfolios.

**Interview line:** "The SCR for this motor portfolio is approximately [X]% of earned premium, meaning for every $100 of premium collected, we need $[X] of capital to meet the 99.5% confidence requirement. This is consistent with typical motor SCR intensities of 10-20%."

In [ ]:
# YOUR CODE: Compute SCR, express as % of expected loss and % of premium

## Step 6.4: Confidence Interval on VaR Estimate

**What to do:**
1. Run the FULL simulation 20 times (different random seeds)
2. Record the VaR(99.5%) from each run
3. Compute: mean VaR, std of VaR, 95% CI for VaR

```python
var_estimates = []
for seed in range(20):
    np.random.seed(seed)
    # ... your simulation ...
    var_estimates.append(np.percentile(sim_losses, 99.5))
print(f"VaR 95% CI: [{np.mean(var_estimates) - 1.96*np.std(var_estimates):.0f}, "
      f"{np.mean(var_estimates) + 1.96*np.std(var_estimates):.0f}]")
```

**Why:** The VaR itself has estimation uncertainty! With 100K simulations, only 500 are above the 99.5th percentile. The CI shows how precise your estimate is.

**FRM connection:** You studied confidence intervals for VaR. The standard error of a quantile estimate decreases with √n but also depends on the density at the quantile.

**Interview line:** "With 100,000 simulations, the 95% confidence interval on VaR(99.5%) is approximately ±[X]% of the point estimate. This level of precision is sufficient for capital planning but would need 500K+ simulations for regulatory submission."

In [ ]:
# YOUR CODE: Multiple runs to estimate VaR confidence interval

---
# MODULE 7: PARAMETER UNCERTAINTY & ITS IMPACT ON CAPITAL

**Why this matters:** The simulation in Module 5-6 treated λ and α as KNOWN. But they're ESTIMATED from data — they could be wrong. Adding parameter uncertainty typically increases the SCR by 10-30%.

**FRM Connection:** This is "model risk" — the risk that your model parameters are misspecified. In VaR terms, you're estimating the VaR of a distribution whose parameters you don't know exactly.

---

## Step 7.1: Quantify Parameter Uncertainty from GLM

**What to do:**
1. From your frequency model, get the standard error of the intercept: `freq_model.bse['const']`
2. The "uncertainty" in total expected frequency: approximately `total_expected_claims * freq_model.bse['const']`
3. More precisely: CV of frequency estimate ≈ `1 / np.sqrt(freq_df['ClaimNb'].sum())` (Poisson SE)
4. For severity: CV of mean estimate ≈ `cv_severity / np.sqrt(len(sev_df))`
5. Print both CVs

**Intuition:** With 26,000 claims, the SE of mean severity ≈ SD/√n. With 47,000 total claims, SE of frequency ≈ √λ/√n_policies. These are SMALL for this dataset (~1-2%), but for a smaller book they'd be 10-15%.

**Why it matters for capital:** These CVs become the parameters in a two-stage simulation (next step).

In [ ]:
# YOUR CODE: Compute parameter uncertainty (CV of frequency and severity estimates)

## Step 7.2: Two-Stage Simulation (Process + Parameter Risk)

**What to do:** Run a modified simulation where:
1. **Stage 1 (Parameter Risk):** BEFORE each scenario, perturb the parameters:
   - True λ = λ_estimate × (1 + ε_freq), where ε_freq ~ Normal(0, cv_freq)
   - True mean_sev = mean_sev_estimate × (1 + ε_sev), where ε_sev ~ Normal(0, cv_sev)
   - Or use Gamma distribution for λ: `true_lambda = np.random.gamma(shape_param, scale_param)`
2. **Stage 2 (Process Risk):** Given the perturbed parameters, simulate claims as before

```python
for i in range(n_sim):
    # Stage 1: parameter uncertainty
    true_lambda = np.random.gamma(1/cv_freq**2, lambda_est * cv_freq**2)
    true_mean_sev = np.random.gamma(1/cv_sev**2, mean_sev * cv_sev**2)
    # Stage 2: process risk
    n = np.random.poisson(true_lambda)
    claims = np.random.gamma(alpha, true_mean_sev/alpha, size=n)
    aggregate_losses_2stage[i] = claims.sum()
```

3. Compare VaR(99.5%) WITH parameter uncertainty vs WITHOUT
4. Print the increase (%) — this is the "parameter risk loading"

**Interview line:** "Parameter uncertainty adds approximately [X]% to the capital requirement. This demonstrates why model validation and data quality directly impact capital efficiency — reducing estimation uncertainty reduces the SCR."

In [ ]:
# YOUR CODE: Two-stage simulation, compare VaR with/without parameter uncertainty

---
# MODULE 8: COPULAS & MULTI-LINE DEPENDENCY

**Why this matters for Solvency II:** Real companies have multiple lines of business. The SCR for the COMBINED portfolio depends on HOW CORRELATED the lines are. Copulas model this dependency. The diversification benefit can reduce total SCR by 20-35%.

**FRM Connection:** You studied Gaussian copulas, t-copulas, tail dependence. The 2008 crisis showed Gaussian copulas underestimate tail dependency. Same lesson applies to insurance.

**Approach:** We'll SPLIT our motor portfolio into sub-portfolios (by region or risk segment) to demonstrate multi-line modelling.

---

## Step 8.1: Create Sub-Portfolios (Simulate Multiple Lines)

**What to do:** Split the freMTPL2 portfolio into 3 "lines" by Area:
1. **Line A:** Rural areas (Area A, B) — low frequency, low severity
2. **Line B:** Suburban areas (Area C, D) — medium
3. **Line C:** Urban areas (Area E, F) — high frequency, high severity

For each line, compute:
- Total expected claims (λ) = sum of pred_freq for policies in that line
- Average severity (from sev_data filtered by area)
- Total pure premium

Store these as a dictionary:
```python
lines = {
    'Rural': {'lambda': ..., 'mean_sev': ..., 'premium': ...},
    'Suburban': {'lambda': ..., 'mean_sev': ..., 'premium': ...},
    'Urban': {'lambda': ..., 'mean_sev': ..., 'premium': ...}
}
```

In [ ]:
# YOUR CODE: Split portfolio into 3 lines, compute parameters for each

## Step 8.2: Simulate Each Line Independently

**What to do:**
1. For each line, run 100K Monte Carlo simulations (same approach as Module 5)
2. Store results: `sim_rural`, `sim_suburban`, `sim_urban` (each is array of 100K values)
3. For each line, print: mean, VaR(99.5%), standalone SCR

**Key concept:** These are INDEPENDENT simulations. If we just add them up (sim_rural + sim_suburban + sim_urban), we get the aggregate ASSUMING INDEPENDENCE (correlation = 0). Real diversification benefit is somewhere between 0 correlation and perfect correlation.

In [ ]:
# YOUR CODE: Simulate each line independently, compute standalone SCRs

## Step 8.3: Gaussian Copula — Introducing Dependency

**What to do:**
1. Define a correlation matrix between the 3 lines:
   ```python
   corr_matrix = np.array([
       [1.0, 0.3, 0.2],   # Rural correlations
       [0.3, 1.0, 0.4],   # Suburban correlations
       [0.2, 0.4, 1.0],   # Urban correlations
   ])
   ```
2. Generate correlated uniform samples using Gaussian Copula:
   ```python
   from scipy.stats import norm
   L = np.linalg.cholesky(corr_matrix)  # Cholesky decomposition
   Z = np.random.standard_normal((100000, 3))  # Independent normals
   corr_Z = Z @ L.T  # Correlated normals
   U = norm.cdf(corr_Z)  # Transform to uniforms [0,1]
   ```
3. Use these correlated uniforms to RE-ORDER your simulated losses:
   ```python
   sorted_rural = np.sort(sim_rural)
   dep_rural = sorted_rural[(U[:, 0] * 99999).astype(int)]
   ```
4. Compute dependent aggregate: `total_dependent = dep_rural + dep_suburban + dep_urban`

**Why Cholesky:** It's the standard way to generate correlated Normal samples. L×L^T = Correlation matrix. You multiply independent normals by L to get correlated normals. Then transform to uniforms via the Normal CDF.

**FRM connection:** This is exactly the Gaussian copula from your credit risk chapter! Same math, different application.

In [ ]:
# YOUR CODE: Gaussian copula — generate correlated losses

## Step 8.4: Compute Diversification Benefit

**What to do:**
1. Sum of standalone SCRs: `scr_rural + scr_suburban + scr_urban`
2. Diversified SCR: `VaR(99.5%) of total_dependent - mean(total_dependent)`
3. Diversification Benefit = Sum_standalone - Diversified_SCR
4. Diversification Benefit % = Benefit / Sum_standalone
5. Also compare to perfectly INDEPENDENT case: `total_independent = sim_rural + sim_suburban + sim_urban`

Print:
```
Sum of Standalone SCRs:    $XX.XM (no diversification)
Diversified SCR (Gaussian): $XX.XM (with ρ=0.2-0.4)
Independent SCR:            $XX.XM (ρ=0)
Diversification Benefit:    XX% reduction
```

**Interview line:** "The diversification benefit for this portfolio is approximately [X]%, reducing total SCR from $[A]M (sum of standalone) to $[B]M. This benefit arises because the three sub-portfolios (rural, suburban, urban) have correlations of 0.2-0.4 — meaning it's unlikely all three experience worst-case losses simultaneously."

In [ ]:
# YOUR CODE: Compute diversification benefit

## Step 8.5: t-Copula — Tail Dependence

**What to do:**
1. Repeat Step 8.3 but with a t-copula instead of Gaussian:
   ```python
   from scipy.stats import t as t_dist
   df = 4  # degrees of freedom (lower = more tail dependence)
   
   Z = np.random.standard_normal((100000, 3))
   corr_Z = Z @ L.T
   chi2 = np.random.chisquare(df, size=100000)
   t_samples = corr_Z * np.sqrt(df / chi2)[:, np.newaxis]
   U_t = t_dist.cdf(t_samples, df)
   ```
2. Reorder losses using t-copula uniforms
3. Compute diversified SCR under t-copula
4. Compare: Gaussian SCR vs t-copula SCR

**Key difference:** The t-copula has TAIL DEPENDENCE — when one line has an extreme loss, other lines are MORE likely to also have extreme losses. This means LESS diversification benefit in the tail.

**Interview line:** "I compared Gaussian and t-copulas. The t-copula (df=4) gives approximately [X]% higher SCR than Gaussian because it introduces tail dependence — extreme losses across lines co-occur more often than the Gaussian copula would suggest. This is more realistic for insurance where systemic events (weather, economic shocks) affect multiple segments."

In [ ]:
# YOUR CODE: t-copula simulation, compare SCR to Gaussian copula

---
# MODULE 9: SOLVENCY II STANDARD FORMULA

**Why this matters:** The Standard Formula is what every insurer uses unless they have an approved internal model. Even if you have an internal model, you compare against the Standard Formula to show your internal model is reasonable.

---

## Step 9.1: Standard Formula Parameters for Motor MTPL

**What to do:**
1. Look up prescribed volatility parameters for Motor MTPL (we'll use Solvency II values):
   - σ_premium (Motor MTPL) = 10% (prescribed by EIOPA)
   - σ_reserve (Motor MTPL) = 9.5%
   - Correlation ρ between premium and reserve risk = 0.5
2. Define your volume measures:
   - V_premium = your total earned premium (from Step 4.2)
   - V_reserve = outstanding reserves. Assume 1.5× annual claims (typical for motor: ~18 months average duration)
3. Store these in variables

**Where these come from:** EIOPA (European Insurance and Occupational Pensions Authority) publishes these parameters. They're calibrated from industry-wide European data. Companies can apply for "Undertaking-Specific Parameters" (USP) using their own data.

**Interview line:** "The Standard Formula prescribes σ_premium = 10% for Motor MTPL, calibrated by EIOPA from EU-wide data. If our portfolio is less volatile than average, we could apply for USP to reduce our capital requirement — which is exactly what my internal model can demonstrate."

In [ ]:
# YOUR CODE: Define Standard Formula parameters for Motor MTPL

## Step 9.2: Compute Standard Formula SCR (Non-Life)

**What to do:** Apply the Standard Formula:
1. Compute combined σ:
   ```python
   V_total = V_prem + V_res
   variance = (sigma_prem * V_prem)**2 + (sigma_res * V_res)**2 + 2*rho*sigma_prem*V_prem*sigma_res*V_res
   sigma_combined = np.sqrt(variance) / V_total
   ```
2. Compute SCR:
   ```python
   SCR_standard_formula = 3 * sigma_combined * V_total
   ```
   (The factor 3 approximates VaR(99.5%) for a lognormal distribution)
3. Print: V_prem, V_res, σ_combined, SCR_standard_formula
4. Compare to your internal model SCR from Module 6

**Why factor = 3:** If losses are lognormally distributed, VaR(99.5%) ≈ mean × exp(2.58σ - σ²/2). For small σ (<20%), this approximates to ≈ mean × (1 + 3σ). So SCR ≈ 3 × σ × Volume.

**Interview line:** "The Standard Formula gives SCR = $[X]M. My internal model gives $[Y]M. The [higher/lower] internal model result reflects [reason: our portfolio is more/less volatile than the industry average used to calibrate the standard formula]."

In [ ]:
# YOUR CODE: Standard Formula SCR calculation

## Step 9.3: Internal Model vs Standard Formula — Comparison Table

**What to do:** Create a comparison DataFrame:
```
Metric                    | Standard Formula | Internal Model | Difference
─────────────────────────────────────────────────────────────────────────
SCR ($M)                  | XX.X             | XX.X           | XX.X%
SCR / Premium (%)         | XX.X%            | XX.X%          | 
Implied volatility (σ)    | 10.0% (prescribed)| XX.X% (fitted) |
Diversification benefit   | N/A (single line) | XX%            |
Tail assumption           | Lognormal (approx)| Compound P-G   |
Parameter uncertainty     | Not explicit      | Included (+X%) |
```

**Interview line:** "The internal model gives a [X]% [higher/lower] SCR than the Standard Formula. This is because [reason]. An internal model captures portfolio-specific features — tail shape, dependency structure, parameter uncertainty — that the Standard Formula, by design, cannot."

In [ ]:
# YOUR CODE: Create comparison table — Standard Formula vs Internal Model

## Step 9.4: Sensitivity Analysis — What Drives the SCR?

**What to do:** Test how SCR changes when you vary inputs:
1. **Frequency +20%:** Multiply λ by 1.2, re-simulate, compute new VaR(99.5%)
2. **Severity +20%:** Multiply mean_sev by 1.2, re-simulate
3. **Tail heavier (ξ increased):** Use Lognormal with higher CV
4. **Correlation +50%:** Increase copula correlations
5. **Volume +30%:** More policies

Create a tornado/waterfall chart showing the impact of each on SCR.

**Why:** Regulators and management want to know: "What scenario most threatens our capital position?" This sensitivity analysis answers that.

**Interview line:** "The SCR is most sensitive to severity assumptions — a 20% increase in mean severity increases SCR by approximately [X]%. This means getting the severity model right is the highest priority for capital accuracy. Frequency is second-most important."

In [ ]:
# YOUR CODE: Sensitivity analysis — vary each input, record SCR change

---
# MODULE 10: CAPITAL ALLOCATION & PERFORMANCE MEASUREMENT

**Why this matters for Solvency II:** Capital must be ALLOCATED back to business lines to measure performance. A line that uses $30M of capital must earn enough to justify that capital. This closes the loop: Capital → Pricing → Profitability.

---

## Step 10.1: Euler (Marginal) Capital Allocation

**What to do:**
1. From your copula simulation (Module 8), you have: `dep_rural`, `dep_suburban`, `dep_urban`, and `total_dependent`
2. Find the simulations where total loss is near VaR(99.5%) — use a band (99% to 99.9%):
   ```python
   var_level = np.percentile(total_dependent, 99.5)
   band_lower = np.percentile(total_dependent, 99.0)
   band_upper = np.percentile(total_dependent, 99.9)
   mask = (total_dependent >= band_lower) & (total_dependent <= band_upper)
   ```
3. For each line, compute its average contribution GIVEN total is near VaR:
   ```python
   alloc_rural = dep_rural[mask].mean()
   alloc_suburban = dep_suburban[mask].mean()
   alloc_urban = dep_urban[mask].mean()
   ```
4. Normalize so allocations sum to total SCR
5. Print: allocated capital per line, % of total

**Why Euler:** It's the only allocation method that is:
- Additive (allocations sum to total SCR)
- Risk-sensitive (riskier lines get more capital)
- Marginal (based on contribution to TOTAL risk, not standalone risk)

**Interview line:** "I use the Euler allocation method because it's the only allocation that satisfies the full-allocation property and captures marginal risk contribution. The urban sub-portfolio receives [X]% of capital despite being [Y]% of premium, because it contributes disproportionately to tail risk."

In [ ]:
# YOUR CODE: Euler capital allocation across sub-portfolios

## Step 10.2: Return on Risk-Adjusted Capital (RORAC)

**What to do:**
1. For each line, compute expected profit:
   - Profit = Gross Premium − Expected Loss − Expenses (assume 25% expense ratio)
   - `profit_line = premium_line * (1 - expected_LR - 0.25)`
2. RORAC = Profit / Allocated Capital
3. Compare RORAC across lines

**What to look for:**
- If all lines have similar RORAC → capital is well-priced
- If Urban has low RORAC → it's not earning enough to justify its capital usage → should reprice or de-risk
- If Rural has high RORAC → it's earning more than needed → grow this segment

**This is the ultimate business insight from capital modelling!**

**Interview line:** "RORAC analysis shows the urban sub-portfolio earns [X]% return on its allocated capital versus [Y]% for rural. This suggests we should either increase urban pricing to reflect its capital intensity, or shift growth toward the rural segment for better capital efficiency."

In [ ]:
# YOUR CODE: Compute RORAC for each line, compare

## Step 10.3: Capital Cost Loading in Premium

**What to do:**
1. Assume cost of capital = 10% (shareholders require 10% return on capital)
2. For each line: capital_cost = allocated_capital × 10%
3. Required premium = expected_loss + expenses + capital_cost + target_profit
4. Implied minimum loss ratio = expected_loss / required_premium

**The closed loop:**
```
GLM (Module 2-3) → Expected Loss
Monte Carlo (Module 5-6) → Capital Required
Allocation (Module 10) → Capital per line
Capital Cost → Loading in premium → FEEDS BACK TO PRICING
```

**Interview line:** "My capital model closes the pricing-capital loop. The capital cost loading for the urban segment is approximately $[X] per policy, which must be reflected in the premium. Without this loading, we'd be subsidizing high-risk segments with low-risk segment profits."

In [ ]:
# YOUR CODE: Capital cost loading calculation, complete the feedback loop

---
# MODULE 11: REINSURANCE & ITS IMPACT ON CAPITAL

**Why this matters:** Reinsurance is the primary tool to REDUCE capital requirements. You buy reinsurance to truncate the loss distribution, which reduces VaR(99.5%) significantly.

---

## Step 11.1: Simulate Excess-of-Loss (XoL) Reinsurance

**What to do:**
1. Define XoL treaty: Retention = €5,000 per claim (you pay up to €5K, reinsurer pays the rest)
2. Re-run your Monte Carlo simulation:
   ```python
   for i in range(n_sim):
       n = np.random.poisson(lambda_portfolio)
       claims_gross = np.random.gamma(alpha, mean_sev/alpha, size=n)
       claims_net = np.minimum(claims_gross, 5000)  # Capped at retention
       aggregate_net[i] = claims_net.sum()
       aggregate_gross[i] = claims_gross.sum()
       reinsurance_recovery[i] = aggregate_gross[i] - aggregate_net[i]
   ```
3. Compute: VaR(99.5%) GROSS vs VaR(99.5%) NET
4. Capital SAVING = SCR_gross - SCR_net

**The trade-off:** Reinsurance reduces capital but costs premium. Is the capital saving worth the reinsurance premium?

**Interview line:** "An excess-of-loss treaty with €5K retention reduces the SCR by approximately [X]%, eliminating tail risk from large individual claims. The capital saving of $[Y]M must be weighed against the reinsurance premium to assess net economic value."

In [ ]:
# YOUR CODE: Simulate gross vs net (after XoL reinsurance)

## Step 11.2: Optimal Retention Analysis

**What to do:**
1. Test multiple retention levels: €2K, €5K, €10K, €20K, €50K, unlimited
2. For each, compute: SCR_net, capital_saving, estimated reinsurance premium
   - Simple reinsurance pricing: `reins_premium ≈ E[excess] × (1 + loading)`
   - Loading ≈ 30% is typical
   - `E[excess] = mean of max(claims - retention, 0)`
3. Compute net benefit: `capital_saving × cost_of_capital - reins_premium`
4. Plot: retention on x-axis, net economic benefit on y-axis
5. Find the optimal retention (maximum net benefit)

**Interview line:** "The optimal retention is approximately €[X]K, balancing capital relief against reinsurance cost. Below this retention, the marginal capital saving per euro of reinsurance premium diminishes. This analysis directly informs our reinsurance purchasing strategy."

In [ ]:
# YOUR CODE: Test multiple retentions, find optimal

---
# MODULE 12: FINAL SUMMARY & INTERVIEW PREPARATION

---

## Step 12.1: Executive Summary Dashboard

**What to do:** Create a final summary printout with all key results:

```
══════════════════════════════════════════════════════════════
        SOLVENCY II CAPITAL MODEL — EXECUTIVE SUMMARY
══════════════════════════════════════════════════════════════
Portfolio:
  Policies:           678,013
  Total Exposure:     XXX,XXX policy-years
  Total Claims:       XX,XXX
  Earned Premium:     €XX.XM
  Expected Loss:      €XX.XM (Loss Ratio: XX.X%)

Risk Metrics:
  VaR (99.5%):        €XX.XM
  Expected Shortfall: €XX.XM
  SCR (Internal):     €XX.XM
  SCR (Std Formula):  €XX.XM
  SCR / Premium:      XX.X%

Diversification:
  Sum Standalone SCR: €XX.XM
  Diversified SCR:    €XX.XM
  Benefit:            XX%

Reinsurance Impact:
  Gross SCR:          €XX.XM
  Net SCR (€5K XoL):  €XX.XM
  Capital Saving:     €XX.XM

Key Sensitivities:
  Frequency +20%:     SCR increases XX%
  Severity +20%:      SCR increases XX%
  Tail heavier:       SCR increases XX%
══════════════════════════════════════════════════════════════
```

In [ ]:
# YOUR CODE: Print the complete executive summary with all your results

## Step 12.2: Your Interview Narrative

**Memorize this story — it flows naturally through what you built:**

---

"In this project, I built a complete Solvency II internal capital model for a Motor MTPL portfolio of 678K policies.

**Starting point:** I fitted a Poisson GLM for frequency and a Gamma GLM for severity using 10 risk factors. The models were validated out-of-sample with prediction ratios close to 1.0.

**From pricing to capital:** I used these GLM outputs — the estimated frequency λ and severity parameters — as inputs to a Monte Carlo simulation. The key insight is that pricing gives you the MEAN; capital requires the TAIL. Same distributions, different quantile.

**Aggregate loss:** I simulated 100,000 scenarios of the compound Poisson-Gamma distribution. The 99.5th percentile gives VaR = €[X]M against an expected loss of €[Y]M, implying an SCR of €[Z]M.

**Tail modelling:** I validated that the Gamma body model underestimates the tail. Using GPD (from Extreme Value Theory), I found ξ = [value], confirming heavier tails than Gamma alone suggests. This increases the SCR by approximately [X]%.

**Dependency:** I split the portfolio into rural/suburban/urban sub-portfolios and modelled their dependency using both Gaussian and t-copulas. The diversification benefit is ~[X]%, but the t-copula reduces this by [Y]% because it captures tail dependence.

**Parameter uncertainty:** Adding estimation uncertainty (two-stage simulation) increases SCR by ~[X]%. This demonstrates that better data and models directly reduce capital requirements.

**Standard Formula comparison:** The Standard Formula gives SCR = €[A]M while my internal model gives €[B]M. The difference is because [reason].

**Business impact:** I allocated capital back using the Euler method and computed RORAC by segment. The urban segment has [lower/higher] RORAC, suggesting pricing action. I also showed that a €5K XoL reinsurance treaty reduces SCR by [X]% at a cost of [Y]%, with optimal retention at €[Z]K."

---

**When they ask follow-ups:**
- "Why Poisson?" → It's the natural model for independent event counts. E[N]=Var[N]=λ. If overdispersed, I'd use Negative Binomial.
- "Why Gamma?" → Variance proportional to mean². Flexible shape. If inadequate, supplement with GPD tail.
- "VaR vs ES?" → VaR = threshold, ES = average beyond threshold. ES is coherent (sub-additive). SST uses ES. Solvency II uses VaR.
- "What's the biggest limitation?" → The assumption of stationary parameters. In practice, frequency and severity trend over time (inflation, climate change). A dynamic model would capture this.
- "How would you improve it?" → (1) Add calendar-year trends, (2) use MCMC for parameter uncertainty instead of simple CV, (3) model catastrophe risk separately, (4) add reserving risk from triangle bootstrap.

In [ ]:
# Final cell: Print your key "talking points" numbers
# Fill these in as you complete each module:

print("MY KEY NUMBERS FOR THE INTERVIEW:")
print("="*50)
print(f"Portfolio size: 678,013 policies")
print(f"Claim frequency: [fill in]")
print(f"Mean severity: [fill in]")
print(f"Expected loss (E[S]): [fill in]")
print(f"VaR(99.5%): [fill in]")
print(f"SCR: [fill in]")
print(f"SCR/Premium: [fill in]%")
print(f"Diversification benefit: [fill in]%")
print(f"Parameter uncertainty loading: [fill in]%")
print(f"t-copula vs Gaussian: +[fill in]%")
print(f"Reinsurance capital saving: [fill in]%")
print(f"GPD shape parameter (xi): [fill in]")